# Single Choice Questions

This notebook contains the code to probe the answer of single choice questions from the MentalBench dataset.  

Both models use the default chat template. The Instruct model is directly probed while the reasoning model generates new tokens (default sampling strategy) untill the \<think\> token is reached.  


Results are stored in S3 for later evaluation

In [ ]:
!pip install s3fs

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn.functional as F
import pandas as pd
from tqdm.notebook import tqdm
from google.colab import userdata

END_THINK = 151668
NEWLINE = 271 # two newlines

In [ ]:
think = True

if think:
  model_name = "Qwen/Qwen3-4B-Thinking-2507"
else:
  model_name = "Qwen/Qwen3-4B-Instruct-2507"
device="cuda"

storage_options={
        "key": userdata.get('S3_KEY') ,
        "secret": userdata.get('S3_SECRET'),
        "endpoint_url": userdata.get("S3_HOST")
    }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

In [ ]:
def single_choice(model, tokenizer, question, options, n_options=4, think=False):

    # format the question and options
    model_input = [
        {"role": "system", "content":"You answer single choice questions. Answer truthfully and only with the letter of the correct answer"},
        {"role": "user", "content": question},
        {"role": "user", "content": options},
    ]

    # tokenize input
    tokenized = tokenizer.apply_chat_template(
        model_input,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    ).to(device)

    reasoning = ""
    limit_reached = False

    # querry model
    with torch.no_grad():
      if think:
        # let the model generate tokens untill the thinking process is done
        out_thinking = model.generate(**tokenized, max_new_tokens=1000, stop_strings="</think>", tokenizer=tokenizer)

        # add the thinking end token in case the model was interupted
        if out_thinking[0, -1] != END_THINK:
          out_thinking = F.pad(out_thinking, (0, 1), value=END_THINK)
          limit_reached = True
          print(out_thinking.shape)

        # decode the reasoning chain for later evaluation
        reasoning = tokenizer.decode(out_thinking)[0].split("<|im_start|>assistant\n")[-1]

        # append newlines for consistency
        #out_thinking = torch.cat([out_thinking, tokenizer(["/n/n"], return_tensors="pt").input_ids.to(device)], dim=1)
        out_thinking = F.pad(out_thinking, (0, 1), value=NEWLINE)

        # calculate logits
        out = model(out_thinking)
      else:
        # calculate the logits directly on the input tokens
        out = model(**tokenized)

    # logits for next token
    logits = out.logits[0][-1].to("cpu")

    # tokenize answers (letters only)
    letters = [chr(65+i) for i in range(n_options)]
    letter_id = tokenizer([f" {l}" for l in letters], return_tensors="pt").input_ids[:,0]

    # select option with highest score
    return reasoning, limit_reached, letters[logits[letter_id].argmax()]

In [ ]:
# set to -1 to load original data or to an existing checkpoint
checkpoint = 278

if checkpoint >= 0:

  # load checkpoint from s3
  print(f"loading checkpoint {checkpoint}")
  sample = pd.read_csv(
    f"s3://aisa/results/{model_name}/mb_checkpoint_{checkpoint:04d}.csv",
    storage_options=storage_options)
else:

  # fetch original dataset from huggingface
  print("loading original data")
  df = pd.read_csv("hf://datasets/hysong/MentalBench/mentalbench.csv")

  # sample from each single choice difficulty category 100 samples
  sample = df[df["type"].isin(["type1", "type2", "type4"])].groupby("type").sample(100, random_state=42)
  sample = sample.reset_index(drop=True)


In [ ]:
# main loop
for idx in tqdm(range(checkpoint+1, sample.shape[0])):
  q = sample.loc[idx,"question"]
  o = sample.loc[idx,"option"]
  t, l, a = single_choice(model,tokenizer, q, o, think=think)
  torch.cuda.empty_cache()
  sample.loc[idx, model_name + "_think"] = t
  sample.loc[idx, model_name + "_limmit"] = l
  sample.loc[idx, model_name + "_answer"] = a

  # save checkpoint in s3
  sample.to_csv(
    f"s3://aisa/results/{model_name}/mb_checkpoint_{idx:04d}.csv",
    storage_options=storage_options)
